In [0]:
df=spark.sql("""
          SELECT
    u.usage_date,
    u.sku_name,
    u.usage_metadata,
    u.identity_metadata,
    ROUND(SUM(u.usage_quantity), 2) AS dbus,
    ROUND(SUM(u.usage_quantity * lp.pricing.default), 2) AS list_cost
  FROM system.billing.usage u
  JOIN system.billing.list_prices lp
    ON u.cloud = lp.cloud
   AND u.sku_name = lp.sku_name
   AND u.usage_start_time >= lp.price_start_time
   AND (u.usage_end_time <= lp.price_end_time OR lp.price_end_time IS NULL)
  WHERE u.usage_date >= DATE'2026-04-15'
  GROUP BY
    u.usage_date,
    u.sku_name,
    u.usage_metadata,
    u.identity_metadata
  ORDER BY list_cost DESC
          """)

In [0]:
df.write.mode("overwrite").saveAsTable("amit.default.usage")